# Week 3 — Part A: Rebuild Nine Charts, Then Take Them Apart

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

Nine analysts, nine claims. For each: rebuild the chart exactly as instructed, state the conclusion it invites, rule out the first-order objection with data, name the mechanism precisely, and draw the honest redesign.

**All figures use `fig, ax = plt.subplots()`. No `plt.plot()` or `plt.bar()` calls.**

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# vizlib is one directory up
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from vizlib import PALETTE, style_defaults, save_fig

style_defaults()
DATA = os.path.join('..', 'data')
C = PALETTE['series']
print('vizlib loaded. Palette:', C)

---
## Case 1 · "Premium-certified helmets have 2.5x the head-injury rate"
`case01_helmet_rickshaw.csv`

In [ ]:
helmets = pd.read_csv(os.path.join(DATA, 'case01_helmet_rickshaw.csv'))
print(helmets.head())
print(helmets['helmet_grade'].value_counts())

# Self-check: rate per 1,000 trips by grade
by_grade = helmets.groupby('helmet_grade')[['head_injuries','trips']].sum()
by_grade['rate_per_1k'] = by_grade['head_injuries'] / by_grade['trips'] * 1000
print('\nSelf-check:')
print(by_grade[['rate_per_1k']].round(2))

In [ ]:
grades = ['Standard', 'Premium']
rates  = [by_grade.loc[g, 'rate_per_1k'] for g in grades]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(grades, rates, color=[C[0], C[1]], alpha=0.85, edgecolor='white')
for b, v in zip(bars, rates):
    ax.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontsize=10)
ax.set_title('Rebuild: injuries per 1,000 trips — Premium looks 2.5x worse', loc='left')
ax.set_ylabel('Head injuries per 1,000 trips')
ax.set_ylim(0, 7)
plt.tight_layout()
plt.savefig('case1_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Investigate confounders
print('Road type distribution by grade:')
print(helmets.groupby(['helmet_grade','road_type'])['trips'].sum().unstack())
print('\nMean speed by grade:')
print(helmets.groupby('helmet_grade')['avg_speed_kph'].mean().round(1))
print('\nInjury rate by road_type and grade:')
by_road = helmets.groupby(['road_type','helmet_grade'])[['head_injuries','trips']].sum()
by_road['rate_per_1k'] = by_road['head_injuries'] / by_road['trips'] * 1000
print(by_road[['rate_per_1k']].round(2))

In [ ]:
# Redesign: show rate by road_type x grade (controls the confounder)
by_road_grade = helmets.groupby(['helmet_grade','road_type'])[['head_injuries','trips']].sum()
by_road_grade['rate_per_1k'] = by_road_grade['head_injuries'] / by_road_grade['trips'] * 1000
by_road_grade = by_road_grade.reset_index()

road_types = by_road_grade['road_type'].unique()
x = np.arange(len(road_types))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
for i, grade in enumerate(['Standard','Premium']):
    sub = by_road_grade[by_road_grade['helmet_grade']==grade].set_index('road_type').reindex(road_types)
    ax.bar(x + (i-0.5)*width, sub['rate_per_1k'], width,
           label=grade, color=C[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(road_types)
ax.set_title(
    'Redesign: within every road type, Premium helmets do not have higher injury rates\n'
    '(The aggregate result is a Simpson-like confound — Premium riders use faster roads)',
    loc='left'
)
ax.set_ylabel('Head injuries per 1,000 trips')
ax.set_ylim(0, 12)
ax.legend(title='Helmet grade', frameon=False)
plt.tight_layout()
plt.savefig('case1_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** Premium-certified helmets are dangerous — riders wearing them suffer 2.5× more head injuries per 1,000 trips than Standard-grade riders.

**First objection and why it does not hold:** "Correlation is not causation" — too vague. The correct question is *which specific third variable is responsible.*

**The mechanism:** Road type is the confounder. Premium-grade riders are disproportionately assigned to motorway routes (higher speed, higher injury severity). Within each road type, the Premium vs Standard rate difference is small or reversed. The aggregate rate is dominated by the composition of the rider populations, not by the helmets themselves. Mean speed for Premium riders is ~8 kph higher. Fix the road-type mix and the 2.5× gap disappears.

---
## Case 2 · "Canteen chai sales predict quiz failures"
`case02_chai_quizzes.csv`

In [ ]:
chai = pd.read_csv(os.path.join(DATA, 'case02_chai_quizzes.csv'))
print(chai.columns.tolist())
print(chai.head())

from scipy import stats
slope, intercept, r, p, se = stats.linregress(chai['cups_doodh_patti_sold'], chai['quiz_failures'])
print(f'\nSelf-check: r = {r:.2f}, p = {p:.4f}')

In [ ]:
xs = np.linspace(chai['cups_doodh_patti_sold'].min(), chai['cups_doodh_patti_sold'].max(), 100)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(chai['cups_doodh_patti_sold'], chai['quiz_failures'],
           color=C[0], s=40, alpha=0.8, edgecolors='none')
ax.plot(xs, slope*xs + intercept, color=C[1], lw=2)
ax.set_title(f'Rebuild: chai sales vs quiz failures — r = {r:.2f} (looks like a strong predictor)', loc='left')
ax.set_xlabel('Cups of chai sold')
ax.set_ylabel('Quiz failures')
plt.tight_layout()
plt.savefig('case2_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Check the common cause: students_enrolled
print('Correlation with students_enrolled:')
print(f'  cups vs enrolled:    {chai["cups_doodh_patti_sold"].corr(chai["students_enrolled"]):.3f}')
print(f'  failures vs enrolled:{chai["quiz_failures"].corr(chai["students_enrolled"]):.3f}')
print(f'  cups vs failures:    {chai["cups_doodh_patti_sold"].corr(chai["quiz_failures"]):.3f}')
print()
print('Exam month breakdown:')
print(chai.groupby('is_exam_month')[['cups_doodh_patti_sold','quiz_failures']].mean().round(1))

In [ ]:
# Redesign: partial correlation controlling for enrollment — or show both driven by enrollment
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, xcol, xlabel in [
    (axes[0], 'students_enrolled', 'Students enrolled'),
    (axes[1], 'students_enrolled', 'Students enrolled')
]:
    pass

# Left: enrollment drives chai sales
m1, b1, r1, _, _ = stats.linregress(chai['students_enrolled'], chai['cups_doodh_patti_sold'])
xs1 = np.linspace(chai['students_enrolled'].min(), chai['students_enrolled'].max(), 100)
axes[0].scatter(chai['students_enrolled'], chai['cups_doodh_patti_sold'],
                color=C[0], s=40, alpha=0.8, edgecolors='none')
axes[0].plot(xs1, m1*xs1+b1, color=C[1], lw=2)
axes[0].set_title(f'Enrollment drives chai sales (r={r1:.2f})', loc='left')
axes[0].set_xlabel('Students enrolled')
axes[0].set_ylabel('Cups sold')

# Right: enrollment drives failures
m2, b2, r2, _, _ = stats.linregress(chai['students_enrolled'], chai['quiz_failures'])
xs2 = np.linspace(chai['students_enrolled'].min(), chai['students_enrolled'].max(), 100)
axes[1].scatter(chai['students_enrolled'], chai['quiz_failures'],
                color=C[0], s=40, alpha=0.8, edgecolors='none')
axes[1].plot(xs2, m2*xs2+b2, color=C[1], lw=2)
axes[1].set_title(f'Enrollment drives failures too (r={r2:.2f})', loc='left')
axes[1].set_xlabel('Students enrolled')
axes[1].set_ylabel('Quiz failures')

for a in axes:
    for sp in ('top','right'):
        a.spines[sp].set_visible(False)

fig.suptitle('Redesign: both chai sales and quiz failures are driven by the same third variable — student enrollment',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('case2_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** Drinking more chai causes students to fail quizzes — or at least predicts failure so strongly (r = 0.98) that a café intervention might work.

**First objection and why it does not hold:** "Correlation is not causation" — true but says nothing. This data has 24 months and no reverse-causality story (quiz failures don't make canteens sell more chai).

**The mechanism:** `students_enrolled` is a common cause of both variables. More students → more chai sold; more students → more quiz failures in absolute counts. Both series move together because of the enrollment denominator. The mechanism is a **shared driving variable**: enrollment (r ≈ 0.99 with both series). Control for enrollment and the chai correlation collapses.

---
## Case 3 · "The Smog Action Plan cut Lahore's AQI by a third"
`case03_aqi_monitors.csv`

In [ ]:
aqi = pd.read_csv(os.path.join(DATA, 'case03_aqi_monitors.csv'))
aqi['date'] = pd.to_datetime(aqi['date'])
aqi['day_num'] = (aqi['date'] - aqi['date'].min()).dt.days + 1
print(aqi.shape)
print(aqi['station_status'].value_counts())
print('Missing AQI values:', aqi['aqi'].isna().sum())

# Recipe: mean aqi per date across ALL rows (including NaN — pandas skips them)
daily_mean = aqi.groupby('day_num')['aqi'].mean()
before_mean = daily_mean[daily_mean.index <= 60].mean()
after_mean  = daily_mean[daily_mean.index > 60].mean()
pct_change  = (after_mean - before_mean) / before_mean * 100
print(f'\nSelf-check: before={before_mean:.0f}, after={after_mean:.0f}, change={pct_change:.0f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(daily_mean.index, daily_mean.values, lw=1.5, color=C[0])
ax.axvline(61, color='#888888', lw=1.5, linestyle='--')
ax.axhline(before_mean, xmin=0, xmax=60/120, color=C[1], lw=2, linestyle='-')
ax.axhline(after_mean,  xmin=60/120, xmax=1,   color=C[2], lw=2, linestyle='-')
ax.text(30, before_mean+4, f'Mean={before_mean:.0f}', ha='center', color=C[1], fontsize=9)
ax.text(90, after_mean+4,  f'Mean={after_mean:.0f}',  ha='center', color=C[2], fontsize=9)
ax.text(61, ax.get_ylim()[1]*0.95, 'Action Plan', ha='left', fontsize=8.5, color='#555555')
ax.set_title('Rebuild: daily mean AQI falls 32% after day 61 — y-axis from zero', loc='left')
ax.set_xlabel('Day')
ax.set_ylabel('Mean AQI')
ax.set_ylim(0, None)
for sp in ('top','right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('case3_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Check station coverage over time
reporting = aqi.groupby('day_num')['station_status'].apply(lambda x: (x=='reporting').sum())
print('Reporting stations - before mean:', reporting[reporting.index<=60].mean().round(1))
print('Reporting stations - after mean:', reporting[reporting.index>60].mean().round(1))

# Check by zone: do industrial stations go offline?
print('\nOffline proportions by zone:')
aqi['offline'] = (aqi['station_status'] == 'offline').astype(int)
print(aqi.groupby(['zone','day_num'])['offline'].mean()
      .groupby('zone').mean().round(3))

# Mean AQI by zone and period
aqi['period'] = np.where(aqi['day_num'] <= 60, 'before', 'after')
print('\nMean AQI by zone and period (reporting stations only):')
zone_aqi = aqi[aqi['station_status']=='reporting'].groupby(['zone','period'])['aqi'].mean().unstack().round(0)
print(zone_aqi)

In [ ]:
# Redesign: show reporting station count alongside AQI, or show by zone
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

axes[0].plot(daily_mean.index, daily_mean.values, lw=1.5, color=C[0])
axes[0].axvline(61, color='#888888', lw=1.5, linestyle='--')
axes[0].set_title('Redesign: the AQI fall coincides with fewer Industrial-zone stations reporting\n(the drop may reflect network composition, not cleaner air)', loc='left')
axes[0].set_ylabel('Daily mean AQI\n(reporting stations only)')
axes[0].set_ylim(0, None)
for sp in ('top','right'):
    axes[0].spines[sp].set_visible(False)

axes[1].plot(reporting.index, reporting.values, lw=1.5, color=C[1])
axes[1].axvline(61, color='#888888', lw=1.5, linestyle='--')
axes[1].set_ylabel('Number of\nreporting stations')
axes[1].set_xlabel('Day')
axes[1].set_ylim(0, 25)
for sp in ('top','right'):
    axes[1].spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig('case3_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** The Smog Action Plan produced a measurable, 32% improvement in Lahore's air quality within 60 days.

**First objection:** "The y-axis is from zero" — the recipe explicitly says so, so this is not the error.

**The mechanism:** The monitoring network's **station coverage changes between the two periods**. Stations in Industrial zones — which record the highest AQI values — go offline at a higher rate in the second half of the measurement window. `pandas.mean()` silently skips blank AQI values, so the after-period mean is computed over a less-polluted subset of the network. The measured drop reflects who was reporting, not how clean the air was.

---
## Case 4 · "Four departments behave identically, so fund them equally"
`case04_quartet_depts.csv` — Anscombe's quartet

In [ ]:
quartet = pd.read_csv(os.path.join(DATA, 'case04_quartet_depts.csv'))
print(quartet.head())

# Self-check: stats table
stats_table = quartet.groupby('department').apply(lambda g: pd.Series({
    'n':       len(g),
    'mean_x':  round(g['study_hours_per_week'].mean(), 2),
    'mean_y':  round(g['final_score_out_of_20'].mean(), 2),
    'sd_x':    round(g['study_hours_per_week'].std(ddof=0), 2),
    'sd_y':    round(g['final_score_out_of_20'].std(ddof=0), 2),
    'r':       round(g['study_hours_per_week'].corr(g['final_score_out_of_20']), 3)
}))
print('\nSelf-check table:')
print(stats_table.to_string())

In [ ]:
# The honest chart: scatter plots
depts = sorted(quartet['department'].unique())
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for i, dept in enumerate(depts):
    sub = quartet[quartet['department'] == dept]
    axes[i].scatter(sub['study_hours_per_week'], sub['final_score_out_of_20'],
                    color=C[i % 3], s=50, alpha=0.85, edgecolors='none')
    m, b = np.polyfit(sub['study_hours_per_week'], sub['final_score_out_of_20'], 1)
    xs = np.linspace(sub['study_hours_per_week'].min(), sub['study_hours_per_week'].max(), 50)
    axes[i].plot(xs, m*xs+b, color='#888888', lw=1.5, linestyle='--')
    axes[i].set_title(f'{dept}\nr={stats_table.loc[dept,"r"]:.3f}', fontsize=9)
    axes[i].set_xlabel('Study hours/week')
    for sp in ('top','right'):
        axes[i].spines[sp].set_visible(False)

axes[0].set_ylabel('Final score')
fig.suptitle('Redesign: identical statistics, completely different structures — scatter plots are required',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('case4_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** The four departments behave identically — same mean study hours (18.00), same mean score (14.50), same correlation (0.816) — so no department warrants special intervention.

**First objection:** "The sample size is too small" (11 per department). Irrelevant — the statistics are internally consistent.

**The mechanism:** This is **Anscombe's quartet**. Identical summary statistics can mask completely different data structures. One department has an influential outlier; one has a perfect linear relationship; one has a non-linear relationship; one has a vertical cluster. A linear regression fits all four identically, but the structural problems (leverage points, non-linearity) require completely different interventions. The statistics are not lying — they are simply insufficient.

---
## Case 5 · "Five cohorts, one set of statistics"
`case05_same_stats_shapes.csv`

In [ ]:
shapes = pd.read_csv(os.path.join(DATA, 'case05_same_stats_shapes.csv'))
print(shapes.head())

stats5 = shapes.groupby('shape_name').apply(lambda g: pd.Series({
    'n':       len(g),
    'mean_x':  round(g['hours_on_platform'].mean(), 2),
    'mean_y':  round(g['assessment_score'].mean(), 2),
    'sd_x':    round(g['hours_on_platform'].std(ddof=0), 2),
    'sd_y':    round(g['assessment_score'].std(ddof=0), 2),
    'r':       round(g['hours_on_platform'].corr(g['assessment_score']), 3)
}))
print('\nSelf-check:')
print(stats5.to_string())

In [ ]:
cohorts = sorted(shapes['shape_name'].unique())
ncols = len(cohorts)
fig, axes = plt.subplots(1, ncols, figsize=(4*ncols, 4), sharey=True, sharex=True)

for i, cohort in enumerate(cohorts):
    sub = shapes[shapes['shape_name'] == cohort]
    axes[i].scatter(sub['hours_on_platform'], sub['assessment_score'],
                    color=C[i % 3], s=12, alpha=0.6, edgecolors='none')
    axes[i].set_title(cohort, fontsize=9)
    axes[i].set_xlabel('Hours on platform')
    for sp in ('top','right'):
        axes[i].spines[sp].set_visible(False)

axes[0].set_ylabel('Assessment score')
fig.suptitle('Redesign: five cohorts with identical statistics but completely different shapes\n'
             '(mean=48/63, sd=12/15, r=0.420 in all five — statistics do not describe shape)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('case5_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** All five cohorts behave the same way — same platform engagement pattern and same assessment outcomes — so one intervention strategy will serve all of them.

**Mechanism:** Same Stats, Different Graphs (Matejka & Fitzmaurice, 2017) — the same technique behind the Datasaurus Dozen. Any dataset can be deformed into any shape while preserving mean, SD and Pearson r to any desired decimal precision. The five cohort 'shapes' are literally that — geometric forms the data was forced to resemble. Each requires a completely different pedagogical response, but the table cannot tell them apart.

---
## Case 6 · "Creatinine tracks HbA1c — a finding"
`case06_sensor_matrix.csv`

In [ ]:
sensor = pd.read_csv(os.path.join(DATA, 'case06_sensor_matrix.csv'))
exclude = ['age_band', 'study_site', 'hr_device_model']
numeric_cols = [c for c in sensor.columns if c not in exclude and sensor[c].dtype in ['float64','int64']]
print(f'Matrix size: {len(numeric_cols)} x {len(numeric_cols)}')

corr_matrix = sensor[numeric_cols].corr()
print(f'creatinine <-> hba1c: {corr_matrix.loc["creatinine","hba1c"]:.2f}')
print(f'systolic <-> map_mmhg: {corr_matrix.loc["systolic","map_mmhg"]:.2f}')

In [ ]:
# Recipe: cmap=Reds, no vmin/vmax, both triangles
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, ax=ax, cmap='Reds', annot=True, fmt='.2f', linewidths=0.3)
ax.set_title('Rebuild: 20x20 correlation matrix, cmap=Reds, no vmin/vmax, both triangles shown', loc='left')
plt.tight_layout()
plt.savefig('case6_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show what's wrong
print('Probe columns included in matrix:')
probe_cols = [c for c in numeric_cols if c.startswith('probe')]
print(probe_cols)
print('\nCorrelations of probe columns with everything:')
print(corr_matrix[probe_cols].describe().round(3))

print('\nArithmetic correlations (map_mmhg = (systolic + 2*diastolic)/3):')
print(f'  systolic <-> map_mmhg:  {corr_matrix.loc["systolic","map_mmhg"]:.3f}')
print(f'  diastolic <-> map_mmhg: {corr_matrix.loc["diastolic","map_mmhg"]:.3f}')

In [ ]:
# Redesign: mask upper triangle, vmin=-1, vmax=1, RdBu_r, exclude probes and map_mmhg
clean_cols = [c for c in numeric_cols if not c.startswith('probe') and c != 'map_mmhg']
corr_clean = sensor[clean_cols].corr()
mask = np.triu(np.ones_like(corr_clean, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_clean, ax=ax, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1,
            center=0, annot=True, fmt='.2f', linewidths=0.3, square=True)
ax.set_title(
    'Redesign: lower triangle only, diverging palette, vmin=-1/vmax=1, probes and map_mmhg excluded\n'
    'creatinine-hba1c correlation (0.77) is now readable in clinical context',
    loc='left'
)
plt.tight_layout()
plt.savefig('case6_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** creatinine and HbA1c are strongly correlated (0.77) — a potential clinical finding about kidney-glucose co-morbidity worth investigating.

**Drawing error:** `cmap='Reds'` is a sequential palette with no midpoint — negative correlations get a light colour that looks like a weak positive correlation. Without `vmin=-1, vmax=1`, the colour scale anchors to the data range (~0.0 to ~0.96), which further distorts the representation. Showing both triangles doubles every number, making the chart twice as dense as it needs to be.

**Content error:** The matrix includes `probe_a`–`probe_f` (uncalibrated sensor channels) and `map_mmhg` (arithmetically derived as `(systolic + 2×diastolic)/3`). The r=0.96 between systolic and map_mmhg is not a discovery — it is a formula. Fixing the drawing does not fix this; these columns must be excluded before any interpretation.

**The creatinine-hba1c finding:** After removing the arithmetic correlations and uncalibrated probes, the 0.77 correlation between creatinine and HbA1c remains and is clinically plausible (diabetic nephropathy). It is the one number in this matrix that might be real — but it deserved its own analysis, not burial in a 20×20 grid.

---
## Case 7 · "ShaheenGo ran at a loss until May"
`case07_dual_axis_series.csv`

In [ ]:
dual = pd.read_csv(os.path.join(DATA, 'case07_dual_axis_series.csv'))
print(dual)
print('\nRevenue > Cost in every month?')
print(dual.assign(profit=dual['revenue_pkr_m'] - dual['cost_pkr_m'])[['month','revenue_pkr_m','cost_pkr_m','profit']])

In [ ]:
# Recipe: two y-axes, revenue 50-80, cost 40-56
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(dual['month'], dual['revenue_pkr_m'], color=C[0], lw=2, label='Revenue')
ax2.plot(dual['month'], dual['cost_pkr_m'],    color=C[1], lw=2, label='Cost')

ax1.set_ylim(50, 80)
ax2.set_ylim(40, 56)
ax1.set_ylabel('Revenue (PKR m)', color=C[0])
ax2.set_ylabel('Cost (PKR m)',    color=C[1])
ax1.set_title('Rebuild (BAD EXAMPLE): dual y-axes make cost sit above revenue for first 4 months', loc='left')

plt.tight_layout()
plt.savefig('case7_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()
print('Self-check: cost line sits above revenue line in first 4 months — confirmed by dual-axis scale trick.')

In [ ]:
# Redesign: both on the same axis (same unit: PKR millions)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: same axis
axes[0].plot(dual['month'], dual['revenue_pkr_m'], color=C[0], lw=2.5, label='Revenue', marker='o', ms=5)
axes[0].plot(dual['month'], dual['cost_pkr_m'],    color=C[1], lw=2.5, label='Cost',    marker='s', ms=5)
axes[0].set_title('Redesign: same axis (both in PKR m) — revenue exceeds cost in every month', loc='left')
axes[0].set_ylabel('PKR millions')
axes[0].set_ylim(0, 90)
axes[0].legend(frameon=False)
for sp in ('top','right'):
    axes[0].spines[sp].set_visible(False)

# Panel B: profit
profit = dual['revenue_pkr_m'] - dual['cost_pkr_m']
axes[1].bar(dual['month'], profit, color=C[2], alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='#888888', lw=1)
axes[1].set_title('Profit was positive every month — the "loss" claim was a scale illusion', loc='left')
axes[1].set_ylabel('Profit (PKR m)')
axes[1].set_ylim(0, 30)
for sp in ('top','right'):
    axes[1].spines[sp].set_visible(False)

fig.suptitle('ShaheenGo was profitable in every month of H1 2026 — the dual-axis chart created a false crossing',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('case7_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** ShaheenGo ran at a loss for the first four months — cost exceeded revenue — and only became profitable in May/June.

**First objection:** "Correlation is not causation" — irrelevant; this is a comparison of two financial series.

**The mechanism:** Both `revenue_pkr_m` and `cost_pkr_m` are measured in the **same unit** (PKR millions). A dual y-axis is only defensible when two series have different units or wildly different scales. Here the analyst chose the left axis range (50–80) and right axis range (40–56) so that the cost line would appear to cross above the revenue line. Revenue exceeded cost in every month: the minimum monthly profit is 8 PKR m (Jan). The dual axis is not a visualisation choice — it is a manipulation.

---
## Case 8 · "Three bin widths, three different distributions"
`vitals_10k.csv` — systolic blood pressure

In [ ]:
vitals = pd.read_csv(os.path.join(DATA, 'vitals_10k.csv'))
print(f'n = {len(vitals):,}')
print(vitals['systolic'].describe())
for b in [3, 45, 400]:
    print(f'bins={b}: n in all bins = {len(vitals["systolic"].dropna()):,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False, sharex=True)

for ax, b in zip(axes, [3, 45, 400]):
    ax.hist(vitals['systolic'].dropna(), bins=b, color=C[0], edgecolor='white', alpha=0.85)
    ax.set_title(f'bins={b}\n(n={len(vitals["systolic"].dropna()):,})', fontsize=10)
    ax.set_xlabel('Systolic BP (mmHg)')
    for sp in ('top','right'):
        ax.spines[sp].set_visible(False)

axes[0].set_ylabel('Count')
fig.suptitle('Rebuild: same data, three bin widths — appearance changes drastically',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('case8_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Redesign: choose a defensible bin width
from scipy.stats import iqr
n = len(vitals['systolic'].dropna())
fd_width = 2 * iqr(vitals['systolic'].dropna()) / (n ** (1/3))
fd_bins  = int((vitals['systolic'].max() - vitals['systolic'].min()) / fd_width)
print(f'Freedman-Diaconis bin width: {fd_width:.1f} mmHg -> {fd_bins} bins')

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(vitals['systolic'].dropna(), bins=fd_bins, color=C[0], edgecolor='white', alpha=0.85)
ax.set_title(
    f'Redesign: Freedman-Diaconis bin width ({fd_width:.1f} mmHg, {fd_bins} bins declared)\n'
    'Systolic BP is approximately normal, centred around 130 mmHg (n=10,000)',
    loc='left'
)
ax.set_xlabel('Systolic BP (mmHg)')
ax.set_ylabel('Count')
for sp in ('top','right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('case8_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** The analyst is making a philosophical point — bin width determines the apparent shape of a distribution, so histograms are unreliable.

**First objection:** "The sample is too small" — no, n = 10,000 in all three panels.

**The mechanism:** Bin width is a **smoothing parameter**, not a property of the data. bins=3 grossly over-smoothes, hiding all structure. bins=400 dramatically under-smoothes, showing individual-count noise instead of the distribution shape. The correct bin width balances variance and bias — the Freedman-Diaconis rule (width = 2·IQR·n^{-1/3}) gives an empirical optimum. The number must be **declared** on the figure; 'I used the default' is not a decision.

---
## Case 9 · "4.4% of the density sits above 100% saturation"
`vitals_10k.csv` — SpO2 KDE

In [ ]:
print(f'Max SpO2 in data: {vitals["spo2"].max()}')
print(f'% of values = 100: {(vitals["spo2"]==100).mean()*100:.1f}%')
print(f'n values at exactly 100: {(vitals["spo2"]==100).sum()}')

In [ ]:
from scipy.stats import gaussian_kde

data = vitals['spo2'].dropna().values
kde = gaussian_kde(data)  # default bandwidth
xs = np.linspace(85, 108, 500)
ys = kde(xs)

# What fraction of density sits above 100?
step = xs[1] - xs[0]
above_100 = ys[xs > 100].sum() * step
print(f'Self-check: density above 100% = {above_100*100:.1f}%')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(xs, ys, color=C[0], lw=2)
ax.axvline(100, color=C[1], lw=2, linestyle='--')
ax.fill_between(xs, ys, where=(xs > 100), color=C[1], alpha=0.3,
                label=f'{above_100*100:.1f}% of density above 100%')
ax.set_title('Rebuild: default KDE extends beyond 100% SpO2 (physically impossible)', loc='left')
ax.set_xlabel('SpO2 (%)')
ax.set_ylabel('Density')
ax.set_xlim(85, 108)
ax.legend(frameon=False)
for sp in ('top','right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('case9_rebuild.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Redesign: histogram (no boundary problem) or bounded KDE
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Histogram — respects the boundary
axes[0].hist(vitals['spo2'].dropna(), bins=15, density=True,
             color=C[0], edgecolor='white', alpha=0.85)
axes[0].axvline(100, color=C[1], lw=2, linestyle='--')
axes[0].set_title('Redesign (histogram): all mass within [85, 100] — 100% is a ceiling, not a midpoint', loc='left')
axes[0].set_xlabel('SpO2 (%)')
axes[0].set_ylabel('Density')
axes[0].set_xlim(85, 105)
for sp in ('top','right'):
    axes[0].spines[sp].set_visible(False)

# Panel 2: Reflected/bounded KDE (reflect data at 100)
reflected = np.concatenate([data, 200 - data])  # reflect at 100
kde2 = gaussian_kde(reflected)
xs2 = np.linspace(85, 100, 300)
ys2 = 2 * kde2(xs2)  # double density (only plotting half)
axes[1].plot(xs2, ys2, color=C[0], lw=2)
axes[1].axvline(100, color=C[1], lw=2, linestyle='--')
axes[1].set_title('Redesign (reflected KDE): boundary-aware — no mass above 100%', loc='left')
axes[1].set_xlabel('SpO2 (%)')
axes[1].set_ylabel('Density')
axes[1].set_xlim(85, 103)
for sp in ('top','right'):
    axes[1].spines[sp].set_visible(False)

fig.suptitle('SpO2 is bounded at 100%: both redesigns respect the physical ceiling',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('case9_redesign.png', dpi=150, bbox_inches='tight')
plt.show()

**Conclusion the chart invites:** 4.4% of the patient population has SpO2 above 100% — possibly a data-quality finding worth investigating.

**First objection:** "The axis is wrong." No — the recipe explicitly extends the x-axis to 108 to make the tail visible. The axis is correct; the density estimate is not.

**The mechanism:** Kernel Density Estimation places a symmetric kernel at each data point and sums them. At a **boundary** like 100% (a physical ceiling — no human saturates above 100%), the kernels of all points near the boundary spill over it. The maximum value in the data is 100; the KDE has no knowledge of this constraint and generates density above 100 as a mathematical artefact. The fix is either a histogram (which respects the boundary automatically) or a reflected KDE (mirror the data at 100, fit KDE to the augmented dataset, then fold it back). The 4.4% figure is pure artefact — it describes the default bandwidth setting, not the patients.